<a href="https://colab.research.google.com/github/asukaa2/B-LoRA/blob/main/B_LoRA_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# B-LoRA — Training Demo (Python 3.13-ready)

This notebook trains a **B-LoRA** ([Implicit Style-Content Separation using B-LoRA, CVPR 2024](https://arxiv.org/abs/2403.14572))
from a single image with SDXL, and then shows how to use the two learned blocks:

| LoRA block | Location in the SDXL UNet | Captures | Example usage |
|---|---|---|---|
| **Content B-LoRA** | `up_blocks.0.attentions.0` | the *subject* of the image | re-render the subject in new styles |
| **Style B-LoRA** | `up_blocks.0.attentions.1` | the *style* of the image | apply the style to new subjects |

The training script has been updated for **Python 3.10–3.13** and the modern
diffusers + PEFT stack (LoRA adapters are injected via `peft.LoraConfig`, targeting
exactly the same two attention blocks as the original implementation).

⏱️ **Quick mode** below trains a reasonable B-LoRA in **~30–40 min on a free T4 GPU**;
the original paper settings (1024 px, 1000 steps) are available as `MODE = "paper"` (~2–3 h on T4).

## 1️⃣ Runtime setup
Before running anything: **Runtime → Change runtime type → T4 GPU**.
This cell verifies that a GPU is attached.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2️⃣ Get the code and dependencies
Colab already ships a recent `torch` (>= 2.6), so `requirements.txt` only adds the
diffusers / PEFT / accelerate stack and small utilities.

In [ ]:
!git clone https://github.com/asukaa2/B-LoRA.git
%cd B-LoRA
!pip install -q -r requirements.txt

In [ ]:
import os, shutil, subprocess, sys
sys.path.insert(0, ".")

import torch
import matplotlib.pyplot as plt
from PIL import Image

from blora_utils import BLOCKS, filter_lora, scale_lora, load_lora_state_dict
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

def plot_images(images, titles=None, figsize=(15, 5)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    for i, img in enumerate(images):
        axes[i].imshow(img)
        if titles:
            axes[i].set_title(titles[i])
        axes[i].axis('off')
    plt.show()

print("Environment ready.")

## 3️⃣ Pick a training image
By default we use the paper's canonical teddybear image (already in the repo).
**To train on your own image(s)**: run the upload cell below and place 1–5 images
showing the same subject (different viewpoints/backgrounds work best), then adjust
`INSTANCE_PROMPT` accordingly.

In [ ]:
DATA_DIR = "data/teddybear"
OUTPUT_DIR = "blora-teddybear"
INSTANCE_PROMPT = "A [v] teddybear"   # `A [v]` + a short subject description

os.makedirs(DATA_DIR, exist_ok=True)
if not os.listdir(DATA_DIR):
    shutil.copy("docs/teddybear.jpg", os.path.join(DATA_DIR, "instance.jpg"))

Image.open(os.path.join(DATA_DIR, "instance.jpg"))

In [ ]:
# OPTIONAL — train on your own image(s) instead:
# from google.colab import files
# DATA_DIR = "data/mine"
# OUTPUT_DIR = "blora-mine"
# INSTANCE_PROMPT = "A [v] <your subject>"
# os.makedirs(DATA_DIR, exist_ok=True)
# for name, data in files.upload().items():
#     with open(os.path.join(DATA_DIR, name), "wb") as f:
#         f.write(data)

## 4️⃣ Configure training

| Mode | Resolution | Steps | Time on T4 | Notes |
|---|---|---|---|---|
| `quick` | 768 | 300 | ~30–40 min | good enough to see content/style separation |
| `paper` | 1024 | 1000 | ~2–3 h (T4) / ~1 h (L4) | original paper settings |

Both modes train **rank-64** LoRA adapters on only the two B-LoRA blocks
(≈13 M trainable parameters — this is why the checkpoint is only ~110 MB).

In [ ]:
MODE = "quick"   # "quick" or "paper"

PRESETS = {
    "quick": dict(resolution=768,  max_train_steps=300,  checkpointing_steps=150,
                  validation_epochs=200, num_validation_images=2),
    "paper": dict(resolution=1024, max_train_steps=1000, checkpointing_steps=500,
                  validation_epochs=250, num_validation_images=2),
}
assert MODE in PRESETS, "MODE must be 'quick' or 'paper'"
CFG = dict(
    instance_prompt=INSTANCE_PROMPT,
    rank=64,                  # paper setting
    learning_rate=5e-5,       # paper setting
    seed=0,
    **PRESETS[MODE],
)
CFG

## 5️⃣ Train 🚀
The command below is the one from the README, parameterized by `CFG`.
It uses `accelerate launch` with fp16 mixed precision and 8-bit Adam.

During training the script saves a checkpoint every `checkpointing_steps` steps
(only the LoRA weights, thanks to the custom accelerator hooks) and finally writes
`pytorch_lora_weights.safetensors` into `OUTPUT_DIR`.

In [ ]:
train_cmd = [
    "accelerate", "launch", "train_dreambooth_b-lora_sdxl.py",
    "--pretrained_model_name_or_path", "stabilityai/stable-diffusion-xl-base-1.0",
    "--instance_data_dir", DATA_DIR,
    "--output_dir", OUTPUT_DIR,
    "--instance_prompt", CFG["instance_prompt"],
    "--validation_prompt", CFG["instance_prompt"],
    "--num_validation_images", str(CFG["num_validation_images"]),
    "--validation_epochs", str(CFG["validation_epochs"]),
    "--resolution", str(CFG["resolution"]),
    "--rank", str(CFG["rank"]),
    "--train_batch_size", "1",
    "--learning_rate", str(CFG["learning_rate"]),
    "--lr_scheduler", "constant",
    "--lr_warmup_steps", "0",
    "--max_train_steps", str(CFG["max_train_steps"]),
    "--checkpointing_steps", str(CFG["checkpointing_steps"]),
    "--checkpoints_total_limit", "4",
    "--seed", str(CFG["seed"]),
    "--gradient_checkpointing",
    "--use_8bit_adam",
    "--mixed_precision", "fp16",
]
subprocess.run(train_cmd, check=True)

The trained B-LoRA checkpoint:

In [ ]:
!ls -lh {OUTPUT_DIR}

## 6️⃣ Load SDXL and try the trained B-LoRA
First load the base pipeline (~7 GB download, then ~10 GB VRAM in fp16).

In [ ]:
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=torch.float16,
).to("cuda")

### 6a. Reconstruction — both blocks together
Loading the full checkpoint (content + style blocks) reconstructs the trained subject.

In [ ]:
generator = torch.Generator(device="cuda").manual_seed(48)

trained_sd = load_lora_state_dict(OUTPUT_DIR)      # both blocks
pipeline.load_lora_weights(trained_sd, adapter_name="blora")

images = pipeline(CFG["instance_prompt"], num_images_per_prompt=2, generator=generator).images
plot_images(images, titles=["trained B-LoRA (full)"] * 2)

### 6b. Content block only — text-based stylization
Keep only `up_blocks.0.attentions.0` (the *content* block) and let the text prompt
define the style. `scale_lora` + `filter_lora` do the block selection.

In [ ]:
pipeline.delete_adapters("blora")

content_only = scale_lora(filter_lora(load_lora_state_dict(OUTPUT_DIR), BLOCKS["content"]), 1.0)
pipeline.load_lora_weights(content_only, adapter_name="blora")

images = pipeline("A [v] teddybear made of gold", num_images_per_prompt=2, generator=generator).images
plot_images(images, titles=["content block + text style (gold)"] * 2)

### 6c. Style transfer — your content block + a pretrained style block
Combine the **content** block of your trained B-LoRA with the **style** block of
`lora-library/B-LoRA-pen_sketch` (its style trigger token is `[v30]`).
Tune `content_alpha` / `style_alpha` (0.5–1.5 is a good range) to control the mix.

In [ ]:
def load_b_lora_to_unet(pipe, content_lora_model_id='', style_lora_model_id='',
                        content_alpha=1.0, style_alpha=1.0):
    res_lora = {}
    if content_lora_model_id:
        res_lora.update(scale_lora(filter_lora(load_lora_state_dict(content_lora_model_id), BLOCKS["content"]), content_alpha))
    if style_lora_model_id:
        res_lora.update(scale_lora(filter_lora(load_lora_state_dict(style_lora_model_id), BLOCKS["style"]), style_alpha))
    if "blora" in getattr(pipe.unet, "peft_config", {}):
        pipe.delete_adapters("blora")
    pipe.load_lora_weights(res_lora, adapter_name="blora")

load_b_lora_to_unet(
    pipeline,
    content_lora_model_id=OUTPUT_DIR,                    # your trained content block
    style_lora_model_id="lora-library/B-LoRA-pen_sketch",  # pretrained style block
    content_alpha=1.0, style_alpha=1.1,
)

images = pipeline("A [v] teddybear in [v30] style", num_images_per_prompt=2, generator=generator).images
plot_images(images, titles=["trained content + pen_sketch style"] * 2)

### 6d. Style block only — consistent style generation
Keep only the **style** block and describe any new content in text.

In [ ]:
load_b_lora_to_unet(pipeline, style_lora_model_id=OUTPUT_DIR, style_alpha=1.1)

images = pipeline("A backpack in [v] style", num_images_per_prompt=2, generator=generator).images
plot_images(images, titles=["trained style block on a new subject"] * 2)

# Clean up the adapter when you want to reuse the pipeline for something else:
# pipeline.delete_adapters("blora")
# pipeline.unload_lora_weights()

## 7️⃣ Next steps

* **Train on your own images** — use the upload cell in step 3 and adjust `INSTANCE_PROMPT`
  (the paper uses the template `A [v] <subject>`).
* **Longer training** — switch `MODE = "paper"` in step 4, or resume from a checkpoint with
  `--resume_from_checkpoint latest`.
* **Push to the Hub** — add `--push_to_hub --hub_model_id <user>/<name>` to the training
  command (requires `huggingface-cli login`).
* **Command-line inference** — see `inference.py` and `B_LoRA_inference.ipynb`.

If you use B-LoRA in your research, please cite the paper:

```bibtex
@misc{frenkel2024implicit,
      title={Implicit Style-Content Separation using B-LoRA}, 
      author={Yarden Frenkel and Yael Vinker and Ariel Shamir and Daniel Cohen-Or},
      year={2024},
      eprint={2403.14572},
      archivePrefix={arXiv},
      primaryClass={cs.CV}
}
```